In [ ]:
# Upgrade core data and visualization libraries to their latest versions
%pip install --upgrade pandas numpy seaborn plotly dash

# Kiva Lending Patterns - Assessed Visualizations

Three charts selected as the assessed deliverables for the final submission.

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.express as px

CLEANED_PATH = os.path.join('..', 'data', 'kiva_cleaned.csv')
dfc = pd.read_csv(CLEANED_PATH, index_col=0)
print(f"Cleaned dataset loaded: {dfc.shape[0]:,} rows, {dfc.shape[1]} columns")

### Assessed Chart 1 - Loan Count by Country

**Objective:** Identify which countries have the most Kiva loan records.

**What to look for:** Geographic concentration and the gap between the top country and the rest.

**Limitation:** Count reflects number of loans, not monetary volume.
Country population and economic size affect counts.

In [ ]:
TOP_N_COUNTRIES = 10
country_counts = dfc['country'].value_counts().head(TOP_N_COUNTRIES).reset_index()
country_counts.columns = ['country', 'count']
country_counts['pct'] = (country_counts['count'] / len(dfc) * 100).round(1)

fig_a1 = px.bar(
    country_counts,
    x='count',
    y='country',
    orientation='h',
    text=country_counts['pct'].map('{:.1f}%'.format),
    custom_data=['pct'],
    title=f'Top {TOP_N_COUNTRIES} Countries by Number of Kiva Loans',
    labels={'country': 'Country', 'count': 'Number of Loans'}
)
fig_a1.update_traces(
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Number of loans: %{x:,}<br>'
        '% of total number of loans: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig_a1.show()

### Assessed Chart 2 - Sector Distribution for Top N Countries (100% stacked bar)

**Objective:** Show which sectors account for the largest share of loans in the countries
with the highest lending activity.

**What to look for:** Countries where a single sector dominates vs. countries with a
diversified sector mix. Differences in sector priorities between top-lending countries.

**Configurable parameter:** `TOP_N_COUNTRIES` controls how many countries appear.
Increase to see more context; decrease to focus on the biggest lenders.

**Limitation:** Proportions show sector composition, not monetary volume.
Sample sizes are shown in country labels.

In [ ]:
# Number of top countries to display -- increase or decrease as needed
TOP_N_COUNTRIES = 10

top_countries = dfc['country'].value_counts().head(TOP_N_COUNTRIES).index.tolist()
top_df = dfc[dfc['country'].isin(top_countries)].copy()

sector_by_country = (
    top_df.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
)
country_totals = top_df.groupby('country').size().reset_index(name='total')
sector_by_country = sector_by_country.merge(country_totals, on='country')
sector_by_country['pct'] = sector_by_country['count'] / sector_by_country['total'] * 100
sector_by_country['country_label'] = sector_by_country.apply(
    lambda r: f"{r['country']} ({r['total']:,})", axis=1
)

country_order = (
    top_df.groupby('country').size()
    .sort_values(ascending=False)
    .index.tolist()
)
label_order = [
    sector_by_country.loc[sector_by_country['country'] == c, 'country_label'].iloc[0]
    for c in country_order
]

# Sectors sorted by descending overall share across the full dataset
sector_order = (
    dfc['sector'].value_counts(normalize=True)
    .sort_values(ascending=False)
    .index.tolist()
)

fig_a2 = px.bar(
    sector_by_country,
    x='pct',
    y='country_label',
    color='sector',
    orientation='h',
    custom_data=['pct', 'country'],
    title=f'Sector Distribution for Top {TOP_N_COUNTRIES} Countries by Loan Count',
    labels={
        'pct': '% of Loans',
        'country_label': 'Country',
        'sector': 'Sector'
    },
    category_orders={'country_label': label_order, 'sector': sector_order}
)
fig_a2.update_traces(
    hovertemplate=(
        '<b>%{fullData.name}</b><br>'
        '% of total loans for %{customdata[1]}: %{customdata[0]:.1f}%'
        '<extra></extra>'
    )
)
fig_a2.update_layout(xaxis_ticksuffix='%')
fig_a2.show()

### Assessed Chart 3 - Loan Activity vs Typical Loan Size by Country (bubble scatter)

**Objective:** Explore whether countries with high lending activity tend to have smaller
or larger individual loans, and how much total funding each country attracts.

**X-axis:** Number of loans (lending activity).
**Y-axis:** Median loan amount in USD (typical loan size).
**Bubble size:** Total funded amount in USD (monetary weight of the lending activity).

**Hypothesis to test:** High-activity countries are micro-lending destinations with
smaller loan sizes, while low-activity countries may have larger individual loans.

**Configurable parameter:** `MIN_LOANS` is the minimum number of loans a country must
have to appear. Increase to focus on the most active countries; decrease to include
smaller markets.

**Limitation:** Country-level aggregation hides within-country variation. Bubble size
is total funded amount, not requested amount; partially funded loans affect this figure.

In [ ]:
# Minimum number of loans for a country to appear in the chart -- adjust as needed
MIN_LOANS = 15000

country_bubble = (
    dfc.groupby('country')
    .agg(
        loan_count=('loan_amount', 'count'),
        median_loan_amount=('loan_amount', 'median'),
        total_funded_amount=('funded_amount', 'sum')
    )
    .reset_index()
)

# Dominant sector per country: the sector with the most loans in that country
dominant_sector = (
    dfc.groupby(['country', 'sector'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .drop_duplicates('country')
    .set_index('country')['sector']
)
country_bubble['dominant_sector'] = country_bubble['country'].map(dominant_sector)

country_bubble = (
    country_bubble[country_bubble['loan_count'] >= MIN_LOANS]
    .sort_values('loan_count', ascending=False)
    .reset_index(drop=True)
)

print(f"Countries shown (loan count >= {MIN_LOANS:,}): {len(country_bubble)}")
print(f"Countries excluded (below threshold): "
      f"{dfc['country'].nunique() - len(country_bubble)}")
print("Dominant sectors present:", sorted(country_bubble['dominant_sector'].unique()))

fig_a3 = px.scatter(
    country_bubble,
    x='loan_count',
    y='median_loan_amount',
    size='total_funded_amount',
    color='dominant_sector',
    text='country',
    custom_data=['total_funded_amount', 'dominant_sector'],
    title='Loan Activity vs Typical Loan Size by Country',
    labels={
        'loan_count': 'Number of Loans',
        'median_loan_amount': 'Median Loan Amount (USD)',
        'total_funded_amount': 'Total Funded (USD)',
        'dominant_sector': 'Dominant Sector'
    },
    size_max=60
)
fig_a3.update_traces(
    textposition='middle center',
    textfont_size=10,
    hovertemplate=(
        '<b>%{text}</b><br>'
        'Dominant sector: %{customdata[1]}<br>'
        'Loan count: %{x:,}<br>'
        'Median loan amount: $%{y:,.0f}<br>'
        'Total funded: $%{customdata[0]:,.0f}'
        '<extra></extra>'
    )
)
fig_a3.update_layout(height=750)
fig_a3.show()

---

## Dashboard: Kiva Sector Explorer

Interactive Dash dashboard that complements the 3 assessed charts.
Use the sector dropdown to explore:

- Which countries account for the most loans in the selected sector
- How loan amounts are distributed within the selected sector

Run the cell below, then navigate to **http://localhost:8050** in your browser,
or view inline if your Jupyter environment supports it.

Configurable parameter: `DASH_TOP_N` controls how many top countries the bar chart shows.

In [30]:
import dash
from dash import dcc, html, Input, Output

# Number of top countries shown in the country chart -- adjust as needed
DASH_TOP_N = 15

# Light blue color scheme
BLUE_DEEP   = '#1d4ed8'
BLUE_MID    = '#3b82f6'
SIDEBAR_BG  = '#dbeafe'
PANEL_BG    = '#eff6ff'
TEXT_DARK   = '#1e3a8a'
LABEL_COLOR = '#1e40af'

dash_df = pd.read_csv(os.path.join('..', 'data', 'kiva_cleaned.csv'), index_col=0)

sectors_sorted = sorted(dash_df['sector'].unique())
default_sector = sectors_sorted[0]

app = dash.Dash(
    __name__,
    external_stylesheets=[
        'https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap'
    ]
)

app.layout = html.Div([

    # Sidebar (left)
    html.Div([

        html.H2('Kiva Sector Explorer', style={
            'color': BLUE_DEEP,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '22px',
            'marginTop': '0',
            'marginBottom': '24px',
            'lineHeight': '1.3'
        }),

        html.Label('Select Sector:', style={
            'color': LABEL_COLOR,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '13px',
            'fontWeight': 'bold',
            'display': 'block',
            'marginBottom': '6px'
        }),
        dcc.Dropdown(
            id='sector-dropdown',
            options=[{'label': s, 'value': s} for s in sectors_sorted],
            value=default_sector,
            clearable=False,
            style={'fontFamily': "'Inter', sans-serif", 'fontSize': '14px'}
        ),

        html.Hr(style={'borderColor': BLUE_MID, 'margin': '24px 0'}),

        html.H4('Analysis', style={
            'color': BLUE_DEEP,
            'fontFamily': "'Inter', sans-serif",
            'fontSize': '16px',
            'marginBottom': '10px'
        }),
        html.P(
            '[Placeholder -- analysis text to be added here]',
            style={
                'color': '#64748b',
                'fontFamily': "'Inter', sans-serif",
                'fontSize': '13px',
                'lineHeight': '1.6',
                'fontStyle': 'italic'
            }
        )

    ], style={
        'flex': '0 0 270px',
        'backgroundColor': SIDEBAR_BG,
        'padding': '28px 20px',
        'borderRight': f'2px solid {BLUE_MID}'
    }),

    # Charts (right) - side by side
    html.Div([
        html.Div(dcc.Graph(id='dash-country-bar'),
                 style={'flex': '1', 'minWidth': '0'}),
        html.Div(dcc.Graph(id='dash-loan-dist'),
                 style={'flex': '1', 'minWidth': '0'})
    ], style={
        'flex': '1',
        'display': 'flex',
        'flexDirection': 'row',
        'backgroundColor': '#f8fafc',
        'padding': '4px 8px',
        'alignItems': 'stretch'
    })

], style={
    'display': 'flex',
    'fontFamily': "'Inter', sans-serif",
    'maxWidth': '1400px',
    'margin': '20px auto',
    'boxShadow': '0 4px 14px rgba(0,0,0,0.12)',
    'borderRadius': '6px',
    'overflow': 'hidden',
    'minHeight': '700px'
})


@app.callback(
    Output('dash-country-bar', 'figure'),
    Output('dash-loan-dist', 'figure'),
    Input('sector-dropdown', 'value')
)
def update_charts(selected_sector):
    sector_data = dash_df[dash_df['sector'] == selected_sector]

    country_counts = (
        sector_data['country'].value_counts()
        .head(DASH_TOP_N)
        .reset_index()
    )
    country_counts.columns = ['country', 'count']

    fig_countries = px.bar(
        country_counts,
        x='count',
        y='country',
        orientation='h',
        color_discrete_sequence=[BLUE_MID],
        title=f'Top {DASH_TOP_N} Countries -- {selected_sector}',
        labels={'count': 'Number of Loans', 'country': 'Country'}
    )
    fig_countries.update_layout(
        template='plotly_white',
        plot_bgcolor='#eff6ff',
        yaxis={'categoryorder': 'total ascending'},
        title_font_size=16,
        title_font_family='Inter, sans-serif'
    )

    fig_dist = px.histogram(
        sector_data,
        x='loan_amount',
        nbins=40,
        color_discrete_sequence=[BLUE_DEEP],
        title=f'Loan Amount Distribution -- {selected_sector}',
        labels={'loan_amount': 'Loan Amount (USD)'}
    )
    fig_dist.update_layout(
        template='plotly_white',
        plot_bgcolor='#eff6ff',
        yaxis_title='Number of Loans (log scale)',
        yaxis_type='log',
        yaxis_dtick=1,
        title_font_size=16,
        title_font_family='Inter, sans-serif'
    )

    return fig_countries, fig_dist


app.run(debug=False, port=8050, jupyter_mode='external')


Dash app running on http://127.0.0.1:8050/
